# CSV UMAP Preprocessing Notebook

This notebook demonstrates how to take an arbitrary CSV dataset and prepare it for the RL trajectory viewer. It performs a 2D UMAP projection on numeric features and adds the required administrative columns (`line`, `step`, `action`, `label`).

* Exclude selected columns from projection with `exclude_cols`.
* Optionally choose which columns to use for `step`, `line`, `label`, and `action`.
* Defaults: `step`=0, `line` is the sample index, `action`="".

After running the preprocessing, the resulting CSV can be fed into `preprocess_dataset_generate_knng.py` to produce the final `json.gz` file.


In [ ]:
import pandas as pd
import numpy as np
try:
    import umap
except ImportError:
    raise SystemExit('Missing dependency: umap-learn. Install with `pip install umap-learn`.')


In [ ]:
def preprocess_dataframe(
    df: pd.DataFrame,
    exclude_cols=None,
    label_col=None,
    line_col=None,
    step_col=None,
    action_col=None,
    umap_kwargs=None,
):
    """Project numeric features with UMAP and add admin columns."""
    exclude_cols = set(exclude_cols or [])
    umap_kwargs = umap_kwargs or {}

    if label_col is None:
        for cand in ('label', 'class', 'digit', 'target', 'y'):
            if cand in df.columns:
                label_col = cand
                break
        else:
            label_col = df.columns[0]
    if label_col not in df.columns:
        raise ValueError('label column not found')
    df = df.copy()
    if 'label' not in df.columns:
        df['label'] = df[label_col]

    admin_cols = {'x', 'y', 'label', label_col, line_col or '', step_col or '', action_col or ''}
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    feature_cols = [c for c in numeric_cols if c not in admin_cols and c not in exclude_cols]
    if not feature_cols:
        raise ValueError('No numeric feature columns for UMAP')

    X = df[feature_cols].to_numpy()
    reducer = umap.UMAP(n_components=2, **umap_kwargs)
    embedding = reducer.fit_transform(X)
    df['x'] = embedding[:, 0]
    df['y'] = embedding[:, 1]

    if step_col and step_col in df.columns:
        df['step'] = df[step_col]
    else:
        df['step'] = 0
    if line_col and line_col in df.columns:
        df['line'] = df[line_col]
    else:
        df['line'] = np.arange(len(df), dtype=np.int64)
    if action_col and action_col in df.columns:
        df['action'] = df[action_col]
    else:
        df['action'] = ''

    front = ['line', 'label', 'step', 'action', 'x', 'y']
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
iris_df = iris.frame
processed_iris = preprocess_dataframe(iris_df, label_col='target')
processed_iris.head()


In [ ]:
processed_iris.to_csv('iris_processed.csv', index=False)
print('Wrote iris_processed.csv')


In [17]:
# load csv public/data/fashion-mnist_test.csv
import pandas as pd
fashion = pd.read_csv('public/data/fashion-mnist_test.csv')
processed_fashion_mnist = preprocess_dataframe(
    fashion,
    label_col='label'
)
processed_fashion_mnist.head()

,line,label,step,action,x,y,pixel1,pixel2,pixel3,pixel4,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,,-2.878491,6.373457,0,0,0,0,...,103,87,56,0,0,0,0,0,0,0
1,1,1,0,,-1.301963,-2.433466,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,2,0,,1.726569,9.033625,0,0,0,0,...,0,0,0,0,63,53,31,0,0,0
3,3,2,0,,1.830648,11.039714,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,4,3,0,,-1.102852,7.867284,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# rename / map column headers. currently in format pixel1...pixel784 -> turn into 1x1...28x28 for rowxcol
admin = 6
pix = processed_fashion_mnist.shape[1] - admin
side = int(pix**0.5)
processed_fashion_mnist.columns = (
    list(processed_fashion_mnist.columns[:admin]) +
    [f'{(k//side)+1}x{(k%side)+1}' for k in range(pix)]
)
processed_fashion_mnist.to_csv('public/data/fashion_mnist_processed.csv', index=False)
processed_fashion_mnist.head()

,line,label,step,action,x,y,1x1,1x2,1x3,1x4,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,0,0,0,,-2.878491,6.373457,0,0,0,0,...,103,87,56,0,0,0,0,0,0,0
1,1,1,0,,-1.301963,-2.433466,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,2,0,,1.726569,9.033625,0,0,0,0,...,0,0,0,0,63,53,31,0,0,0
3,3,2,0,,1.830648,11.039714,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,4,3,0,,-1.102852,7.867284,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
import seaborn as sns
penguins = sns.load_dataset('penguins').dropna()
processed_penguins = preprocess_dataframe(penguins, label_col='species', exclude_cols=['sex'])
processed_penguins.head()


In [ ]:
processed_penguins.to_csv('penguins_processed.csv', index=False)
print('Wrote penguins_processed.csv')


Once a CSV has been processed, run the command below in a shell to generate the viewer-ready JSON (gzipped):

```
python preprocess_dataset_generate_knng.py iris_processed.csv iris_processed.json.gz -k 5
```
Replace the filenames as needed for other datasets.
